# Food Delivery Business Analytics

This notebook contains the Python analysis supporting the food delivery business analytics case study.

### Analysis flow
1. Data loading and initial understanding
2. Data quality assessment and preparation
3. Business-question-driven exploratory analysis
4. Statistical validation
5. Driver analysis
6. Customer value and retention analysis
7. Business-facing supporting tables for the Power BI dashboard

### Main libraries
- pandas
- numpy
- matplotlib
- seaborn
- scipy
- scikit-posthocs

> **Note:** Update `DATA_PATH` if the dataset is stored in a different location. The statistical tests and results below reproduce the analysis performed for this project; they are not intended as a generic ML training pipeline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import (
    mannwhitneyu,
    kruskal,
    chi2_contingency,
    chisquare,
    spearmanr,
    fisher_exact
)

# If scikit-posthocs is not installed in your environment, install it once:
# !pip install scikit-posthocs
import scikit_posthocs as sp

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

DATA_PATH = "/content/order_history_kaggle_data.csv"
df = pd.read_csv(DATA_PATH)


In [ ]:
df['Order Status'].value_counts()

In [ ]:
df["Order Ready Marked"].value_counts()

In [ ]:
df['Restaurant ID'].nunique()

In [ ]:
df["Order Placed At"] = pd.to_datetime(df["Order Placed At"])

## 2. Data Quality Assessment & Preparation

In [ ]:
df['Order Status'].isnull().sum()

In [ ]:
df.drop(columns=['Delivery'],inplace=True)

In [ ]:
df['Distance']=df['Distance'].replace("<1km","0.5km")

In [ ]:
df['Distance']=df['Distance'].str.replace("km","")

In [ ]:
df['Distance']=df['Distance'].astype('float')

In [ ]:
df['Items in order'].str.contains(",").sum()/df.shape[0]

In [ ]:
df.drop(columns=['Instructions'],inplace=True)

In [ ]:
df['Discount construct'].isnull().sum()

In [ ]:
df[df["Discount construct"].isna()][[
    "Restaurant discount (Promo)",
    "Restaurant discount (Flat offs, Freebies & others)",
    "Gold discount",
    "Brand pack discount"
]].describe()

In [ ]:
df.drop(columns=['Restaurant penalty (Rejection)'],inplace=True)

In [ ]:
df["Total Discount"] = (
    df["Restaurant discount (Promo)"] +
    df["Restaurant discount (Flat offs, Freebies & others)"] +
    df["Gold discount"] +
    df["Brand pack discount"]
)

In [ ]:
df.drop(columns=['Review'],inplace=True)

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Placed At"]).dt.normalize()

In [ ]:
df[(df['KPT duration (minutes)'].isnull())]['Cancellation / Rejection reason'].value_counts()

In [ ]:
df.drop(columns=['Customer complaint tag'],inplace=True)

In [ ]:
df['Rider wait time (minutes)'].isnull().sum()

In [ ]:
df[(df['Rider wait time (minutes)'].isnull()) & (df['Order Status']=='Delivered') & (df['KPT duration (minutes)'].isnull())].shape

In [ ]:
df.drop(columns='City',inplace=True)

In [ ]:
df.drop(columns='Order Date',inplace=True)

In [ ]:
df["Order Placed At"] = pd.to_datetime(df["Order Placed At"])

df["Order Date"] = df["Order Placed At"].dt.date
df["Order Time"] = df["Order Placed At"].dt.time

df.drop(columns=["Order Placed At"], inplace=True)

In [ ]:
df['Bill subtotal'].isnull().sum()

## 3. Business Questions & Exploratory Analysis

### Do orders cancelled by Zomato exhibit different kitchen preparation times compared with orders that were not cancelled by Zomato?

In [ ]:
df['Zomato_Cancelled'] = df['Cancellation / Rejection reason'].apply(
    lambda x: 'Yes' if x == 'Cancelled by Zomato' else 'No'
)

In [ ]:
test_df = df[df['KPT duration (minutes)'].notna()].copy()

In [ ]:
kpt_summary = (
    test_df
    .groupby('Zomato_Cancelled')
    .agg(
        Orders=('Order ID', 'count'),
        Mean_KPT=('KPT duration (minutes)', 'mean'),
        Median_KPT=('KPT duration (minutes)', 'median'),
        Std_KPT=('KPT duration (minutes)', 'std')
    )
    .round(2)
)

kpt_summary

In [ ]:
# Hypotheses:-

# Null Hypothesis (H₀):
# The distribution of kitchen preparation time is the same for Zomato-cancelled and non-cancelled orders.

# Alternative Hypothesis (H₁):
# The distribution of kitchen preparation time differs between Zomato-cancelled and non-cancelled orders.


In [ ]:
from scipy.stats import mannwhitneyu

kpt_yes = test_df.loc[
    test_df['Zomato_Cancelled'] == 'Yes',
    'KPT duration (minutes)'
]

kpt_no = test_df.loc[
    test_df['Zomato_Cancelled'] == 'No',
    'KPT duration (minutes)'
]

u_stat, p_value = mannwhitneyu(
    kpt_yes,
    kpt_no,
    alternative='two-sided'
)

print(f"U-statistic: {u_stat:.2f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
n1 = len(kpt_yes)
n2 = len(kpt_no)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"Rank-Biserial Correlation: {rank_biserial:.4f}")

### Do orders cancelled by Zomato exhibit different rider wait times compared with orders that were not cancelled by Zomato?

In [ ]:
test_df = df[df['Rider wait time (minutes)'].notna()].copy()

rider_wait_summary = (
    test_df
    .groupby('Zomato_Cancelled')
    .agg(
        Orders=('Order ID', 'count'),
        Mean_Rider_Wait=('Rider wait time (minutes)', 'mean'),
        Median_Rider_Wait=('Rider wait time (minutes)', 'median'),
        Std_Rider_Wait=('Rider wait time (minutes)', 'std')
    )
    .round(2)
)

rider_wait_summary

In [ ]:
from scipy.stats import mannwhitneyu

wait_yes = test_df.loc[
    test_df['Zomato_Cancelled'] == 'Yes',
    'Rider wait time (minutes)'
]

wait_no = test_df.loc[
    test_df['Zomato_Cancelled'] == 'No',
    'Rider wait time (minutes)'
]

u_stat, p_value = mannwhitneyu(
    wait_yes,
    wait_no,
    alternative='two-sided'
)

print(f"U-statistic: {u_stat:.2f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
n1 = len(wait_yes)
n2 = len(wait_no)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"Rank-Biserial Correlation: {rank_biserial:.4f}")

### Is there an association between the Time of Day and orders being cancelled by Zomato?

In [ ]:
df['Order_Hour']=df['Order Time'].apply(lambda x:x.hour)

In [ ]:
def categorize_time(hour):
    if 0 <= hour <= 4:
        return 'Late Night'
    elif 11 <= hour <= 17:
        return 'Lunch/Afternoon'
    elif 18 <= hour <= 23:
        return 'Evening/Dinner'

df['Time_of_Day'] = df['Order_Hour'].apply(categorize_time)

In [ ]:
df['Time_of_Day'].value_counts()

In [ ]:
order_counts = df['Time_of_Day'].value_counts().rename('Total_Orders')
order_counts

In [ ]:
cancelled_orders = df[df['Zomato_Cancelled'] == 'Yes']['Time_of_Day'].value_counts().rename('Zomato_Cancelled')
cancelled_orders

In [ ]:
eda_time = pd.concat([order_counts, cancelled_orders], axis=1).fillna(0)
eda_time
eda_time['Zomato_Cancelled'] = eda_time['Zomato_Cancelled'].astype(int)

In [ ]:
eda_time['Cancellation_Rate (%)'] = (
    eda_time['Zomato_Cancelled'] /
    eda_time['Total_Orders'] * 100
).round(3)

eda_time.sort_values(by='Cancellation_Rate (%)',ascending=False)

In [ ]:
# EDA Interpretation

# The Zomato cancellation rate varied across different time periods of the day. Late Night orders had the highest cancellation rate
#  (0.898%), followed by Evening/Dinner (0.321%) and Lunch/Afternoon (0.275%). Although the cancellation rates differed across these time
#  periods, EDA alone cannot determine whether these differences are statistically significant or simply due to random variation. Therefore,
# a Chi-square Test of Independence was conducted to examine whether Time of Day is associated with Zomato cancellations.

In [ ]:
# Null Hypothesis (H₀): There is no association between the Time of Day and orders being cancelled by Zomato.
# Alternative Hypothesis (H₁): There is an association between the Time of Day and orders being cancelled by Zomato.

In [ ]:
contingency_table = pd.crosstab(
    df['Time_of_Day'],
    df['Zomato_Cancelled']
)

contingency_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square Statistic: {chi2:.4f}")
print(f"P-value: {p:.4f}")
print(f"Degrees of Freedom: {dof}")

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

expected_df

In [ ]:
import numpy as np

n = contingency_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Cramer's V: {cramers_v:.4f}")

In [ ]:
# Although cancellation rates varied across different time periods, the influence of Time of Day on Zomato cancellations was minimal.
# This suggests that Time of Day alone is unlikely to be a major operational driver of cancellations. To achieve a meaningful reduction
# in cancellations, Zomato should prioritize investigating other operational factors that may have a stronger impact.

In [ ]:
df.groupby(['Zomato_Cancelled'])['Total'].mean()

### Is there a significant difference in the total order amount between orders cancelled by Zomato and orders that were not cancelled?

In [ ]:
df.groupby('Zomato_Cancelled')['Total'].agg(['count','mean','median','min','max','std']).round(2)

In [ ]:
df['Total'].skew()

In [ ]:
sns.kdeplot(df['Total'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(
    x='Zomato_Cancelled',
    y='Total',
    data=df
)

plt.xlabel("Zomato Cancelled")
plt.ylabel("Total Order Amount")
plt.title("Total Order Amount by Zomato Cancellation Status")
plt.show()

In [ ]:
sns.histplot(
    data=df,
    x='Total',
    hue='Zomato_Cancelled',
    kde=True
)

In [ ]:
# Since the distribution of Total Order Amount is highly right-skewed, the median provides a more representative measure of central
# tendency than the mean. The median order value for orders not cancelled by Zomato was ₹597.45, compared to ₹544.95 for orders cancelled
# by Zomato. Although the median order values appear different, the number of cancelled orders (81) is substantially smaller than the
# number of non-cancelled orders (21,240). Therefore, the observed difference alone is insufficient to conclude that cancelled and
# non-cancelled orders truly differ in their order values. A statistical test is required to determine whether the observed difference
# in median order values is statistically significant or could have occurred due to random variation.

In [ ]:
# Null Hypothesis (H₀)

# There is no significant difference in the total order amount between orders cancelled by Zomato and orders that were not cancelled.

# Alternative Hypothesis (H₁)

# There is a significant difference in the total order amount between orders cancelled by Zomato and orders that were not cancelled.

In [ ]:
from scipy.stats import mannwhitneyu

cancelled = df[df['Zomato_Cancelled'] == 'Yes']['Total']
not_cancelled = df[df['Zomato_Cancelled'] == 'No']['Total']

u_stat, p_value = mannwhitneyu(
    cancelled,
    not_cancelled,
    alternative='two-sided'
)

print(f"U Statistic : {u_stat:.2f}")
print(f"P-value     : {p_value:.4f}")

In [ ]:
n1 = len(cancelled)
n2 = len(not_cancelled)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"Rank-Biserial Correlation: {rank_biserial:.4f}")

In [ ]:
# Although exploratory analysis showed a slight difference in the median order value between cancelled and non-cancelled orders,
# the Mann–Whitney U test found that this difference was not statistically significant, and the effect size was negligible. This
# suggests that order value is unlikely to be a meaningful factor influencing Zomato cancellations. From a business perspective,
# prioritizing orders solely based on their value is unlikely to significantly reduce cancellation rates. Instead, the business should
# focus on investigating other operational factors that may have a greater impact on cancellations while continuing to maintain a
# consistent customer experience across orders of different values.

### Is there an association between the restaurant subzone and orders being cancelled by Zomato?

In [ ]:
df['Subzone'].value_counts()

In [ ]:
# Total orders received in each subzone
total_orders = (
    df['Subzone']
    .value_counts()
    .rename('Total_Orders')
)

# Zomato-cancelled orders in each subzone
zomato_cancelled = (
    df[df['Zomato_Cancelled'] == 'Yes']['Subzone']
    .value_counts()
    .rename('Zomato_Cancelled')
)

# Combine into one table
subzone_summary = pd.concat(
    [total_orders, zomato_cancelled],
    axis=1
).fillna(0)

# Convert cancelled orders to integer
subzone_summary['Zomato_Cancelled'] = subzone_summary['Zomato_Cancelled'].astype(int)

# Calculate cancellation rate
subzone_summary['Cancellation_Rate (%)'] = (
    subzone_summary['Zomato_Cancelled']
    / subzone_summary['Total_Orders']
    * 100
).round(3)

# Sort by cancellation rate
subzone_summary = subzone_summary.sort_values(
    by='Cancellation_Rate (%)',
    ascending=False
)

subzone_summary

In [ ]:
contingency_table = pd.crosstab(
    df['Subzone'],
    df['Zomato_Cancelled']
)

contingency_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square Statistic:", round(chi2, 4))
print("Degrees of Freedom:", dof)
print("P-value:", round(p_value, 4))

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

expected_df.round(2)

In [ ]:
import numpy as np

n = contingency_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print("Cramér's V:", round(cramers_v, 4))

In [ ]:
df[(df['Cancellation / Rejection reason']=='Cancelled by Customer') & (df['Restaurant compensation (Cancellation)'].isnull())].shape

In [ ]:
temp=df[(df['Cancellation / Rejection reason']=='Cancelled by Customer') & (df['Restaurant compensation (Cancellation)'].isnull()==False)]

In [ ]:
temp[(temp['KPT duration (minutes)'].isnull()) & (temp['Rider wait time (minutes)'].isnull()) & (temp['Order Ready Marked']=='Missed')].shape

In [ ]:
temp[(temp['KPT duration (minutes)'].isnull()==False) & (temp['Rider wait time (minutes)'].isnull()==False)]

In [ ]:
temp[(temp['Rider wait time (minutes)'].isnull()==False) & (temp['KPT duration (minutes)'].isnull())]

In [ ]:
customer_cancel = df[df["Cancellation / Rejection reason"] == "Cancelled by Customer"].copy()


import numpy as np

conditions = [
    (
        customer_cancel["Restaurant compensation (Cancellation)"].isna() &
        customer_cancel["KPT duration (minutes)"].isna() &
        customer_cancel["Rider wait time (minutes)"].isna() &
        (customer_cancel["Order Ready Marked"] == "Missed")
    ),

    (
        customer_cancel["Restaurant compensation (Cancellation)"].notna() &
        customer_cancel["KPT duration (minutes)"].isna() &
        customer_cancel["Rider wait time (minutes)"].isna() &
        (customer_cancel["Order Ready Marked"] == "Missed")
    ),

    (
        customer_cancel["Restaurant compensation (Cancellation)"].notna() &
        customer_cancel["KPT duration (minutes)"].notna() &
        customer_cancel["Rider wait time (minutes)"].isna() &
        (customer_cancel["Order Ready Marked"] == "Missed")
    ),

    (
        customer_cancel["Restaurant compensation (Cancellation)"].notna() &
        customer_cancel["KPT duration (minutes)"].notna() &
        customer_cancel["Rider wait time (minutes)"].notna() &
        (customer_cancel["Order Ready Marked"] == "Correctly")
    ),

    (
        customer_cancel["Restaurant compensation (Cancellation)"].notna() &
        customer_cancel["KPT duration (minutes)"].notna() &
        customer_cancel["Rider wait time (minutes)"].notna() &
        (customer_cancel["Order Ready Marked"] == "Incorrectly")
    )
]

choices = [
    "No Compensation, No Operational Activity",
    "Compensation Only",
    "Preparation Started",
    "Preparation + Rider Wait",
    "Other"
]

customer_cancel["Operational Pattern"] = np.select(
    conditions,
    choices,
    default="Unclassified"
)



In [ ]:
customer_cancel["Operational Pattern"].value_counts()

In [ ]:
customer_cancel.loc[
    customer_cancel["Operational Pattern"] == "Unclassified",
    [
        "Restaurant compensation (Cancellation)",
        "KPT duration (minutes)",
        "Rider wait time (minutes)",
        "Order Ready Marked"
    ]
]

In [ ]:
summary = customer_cancel.groupby("Operational Pattern").agg(
    Orders=("Operational Pattern", "size"),
    Avg_Compensation=("Restaurant compensation (Cancellation)", "mean"),
    Median_Compensation=("Restaurant compensation (Cancellation)", "median"),
    Total_Compensation=("Restaurant compensation (Cancellation)", "sum"),
    Min_Compensation=("Restaurant compensation (Cancellation)", "min"),
    Max_Compensation=("Restaurant compensation (Cancellation)", "max"),
    Std_Compensation=("Restaurant compensation (Cancellation)", "std")
).round(2)

summary

In [ ]:
summary["Percentage"] = (
    summary["Orders"] / summary["Orders"].sum() * 100
).round(1)

summary = summary[
    [
        "Orders",
        "Percentage",
        "Avg_Compensation",
        "Median_Compensation",
        "Total_Compensation",
        "Min_Compensation",
        "Max_Compensation",
        "Std_Compensation"
    ]
]

summary

### Does restaurant compensation differ across different operational patterns of customer-cancelled orders?

In [ ]:
# Null Hypothesis :
# MedianCompensation Only​=MedianPreparation Started​=MedianPreparation + Rider Wait​

# Alternative Hypothesis (H₁)

# At least one operational pattern has a different distribution (or median) of restaurant compensation.

In [ ]:
from scipy.stats import kruskal

test_df = customer_cancel[
    customer_cancel["Operational Pattern"].isin([
        "Compensation Only",
        "Preparation Started",
        "Preparation + Rider Wait"
    ])
]

In [ ]:
comp_only = test_df[
    test_df["Operational Pattern"] == "Compensation Only"
]["Restaurant compensation (Cancellation)"]

prep_started = test_df[
    test_df["Operational Pattern"] == "Preparation Started"
]["Restaurant compensation (Cancellation)"]

prep_rider = test_df[
    test_df["Operational Pattern"] == "Preparation + Rider Wait"
]["Restaurant compensation (Cancellation)"]

In [ ]:
stat, p = kruskal(comp_only, prep_started, prep_rider)

print(f"H Statistic : {stat:.4f}")
print(f"P-value     : {p:.4f}")

In [ ]:
alpha = 0.05

if p < alpha:
    print("Reject the Null Hypothesis")
else:
    print("Fail to Reject the Null Hypothesis")

In [ ]:
N = len(test_df)
k = 3

epsilon_sq = (stat - k + 1) / (N - k)

print(f"Epsilon Squared = {epsilon_sq:.4f}")

### Is Kitchen Preparation Time associated with customer cancellations?

In [ ]:
temp=df.dropna(subset=['KPT duration (minutes)'])

In [ ]:
temp['Cancellation / Rejection reason'].value_counts()

In [ ]:
temp["Customer Cancelled"] = np.where(
    temp["Cancellation / Rejection reason"] == "Cancelled by Customer",
    "Yes",
    "No"
)

In [ ]:
summary = (
    temp.groupby("Customer Cancelled")["KPT duration (minutes)"]
        .agg(
            Count="count",
            Mean="mean",
            Median="median",
            Std="std",
            Min="min",
            Max="max"
        )
        .round(2)
)

summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,5))

sns.boxplot(
    data=temp,
    x="Customer Cancelled",
    y="KPT duration (minutes)"
)

plt.title("Kitchen Preparation Time by Customer Cancellation")
plt.xlabel("Customer Cancelled")
plt.ylabel("KPT Duration (minutes)")
plt.show()

In [ ]:
temp['KPT duration (minutes)'].skew()

In [ ]:
# EDA INTERPRETATION

# The median Kitchen Preparation Time was 14.92 minutes for customer-cancelled orders compared with 16.33
# minutes for non-cancelled orders. The mean Kitchen Preparation Time also remained slightly lower for
# customer-cancelled orders (16.29 minutes) than for non-cancelled orders (17.33 minutes), while the
# variability was similar across both groups. These descriptive results suggest only a small difference in
# Kitchen Preparation Time between the two groups. However, descriptive analysis alone cannot determine
# whether this difference is statistically significant. Therefore, a Mann–Whitney U test was conducted.

In [ ]:
from scipy.stats import mannwhitneyu

cancelled = temp.loc[
    temp["Customer Cancelled"] == "Yes",
    "KPT duration (minutes)"
]

not_cancelled = temp.loc[
    temp["Customer Cancelled"] == "No",
    "KPT duration (minutes)"
]

u_stat, p_value = mannwhitneyu(
    cancelled,
    not_cancelled,
    alternative="two-sided"
)

print(f"U statistic: {u_stat:.2f}")
print(f"p-value: {p_value:.4f}")

In [ ]:
# effect size
n1 = len(cancelled)
n2 = len(not_cancelled)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"Rank-biserial correlation: {rank_biserial:.4f}")

In [ ]:
# The analysis found no statistically significant difference in Kitchen Preparation Time between
# customer-cancelled and non-cancelled orders (Mann–Whitney U = 202,185.50, p = 0.3105). Although
# customer-cancelled orders had slightly lower Kitchen Preparation Times descriptively, the observed
# difference was small in magnitude (rank-biserial correlation = -0.125). Therefore, the findings do
# not provide evidence that Kitchen Preparation Time is meaningfully associated with customer
# cancellations in this dataset.

### Is customer cancellation associated with the time of day in which an order is placed?

In [ ]:
df["Customer Cancelled"] = np.where(
    df["Cancellation / Rejection reason"] == "Cancelled by Customer",
    "Yes",
    "No"
)

df[df['Customer Cancelled']=='Yes']['Time_of_Day'].value_counts()

In [ ]:
import pandas as pd

# Total orders in each time of day
total_orders = (
    df.groupby("Time_of_Day")
      .size()
      .reset_index(name="Total_Orders")
)

# Customer-cancelled orders in each time of day
customer_cancelled = (
    df[df["Customer Cancelled"] == "Yes"]
      .groupby("Time_of_Day")
      .size()
      .reset_index(name="Customer_Cancelled")
)

# Join the two tables
summary = total_orders.merge(
    customer_cancelled,
    on="Time_of_Day",
    how="left"
)

# Replace missing values with 0
summary["Customer_Cancelled"] = (
    summary["Customer_Cancelled"]
    .fillna(0)
    .astype(int)
)

# Calculate cancellation percentage
summary["Cancellation_Percentage"] = (
    summary["Customer_Cancelled"] / summary["Total_Orders"] * 100
).round(2)

summary


In [ ]:
# Null Hypothesis (H₀): There is no association between the time of day and customer
# cancellations.
# Alternative Hypothesis (H₁): There is an association between the time of day and customer
# cancellations.


In [ ]:
contingency_table = pd.crosstab(
    df["Time_of_Day"],
    df["Customer Cancelled"]
)

print("Contingency Table:")
print(contingency_table)

# Chi-Square Test
chi2, p, dof, expected = chi2_contingency(contingency_table)

print("\nChi-Square Test Results")
print(f"Chi-square Statistic : {chi2:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-value              : {p:.4f}")

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

print("\nExpected Frequencies:")
print(expected_df)


In [ ]:
n = contingency_table.to_numpy().sum()
min_dim = min(contingency_table.shape) - 1

cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\nCramér's V : {cramers_v:.4f}")

In [ ]:
# A statistically significant association was observed between the time of day and customer
# cancellations. However, the negligible effect size indicates that, despite differences in
# customer cancellation rates across time periods, the practical influence of time of day on
# customer cancellations is minimal within this dataset.

### Is there an association between the total discount applied to an order and the overall customer rating?

In [ ]:
rated_df = df[df["Rating"].notna()].copy()

In [ ]:
rated_df.groupby("Rating")["Total Discount"].agg(
    Count="count",
    Mean="mean",
    Median="median",
    Min="min",
    Max="max"
).round(2)

In [ ]:
pd.crosstab(
    rated_df["Rating"],
    rated_df["Total Discount"] == 0
)

In [ ]:
# checking normality assumption of total discount in each rating category
from scipy.stats import shapiro

for rating in sorted(rated_df['Rating'].unique()):
    group = rated_df[rated_df['Rating'] == rating]['Total Discount']
    stat, p = shapiro(group)
    print(f"Rating {rating}: p-value = {p:.4f}")

In [ ]:
import matplotlib.pyplot as plt

for rating in sorted(rated_df['Rating'].unique()):
    plt.figure(figsize=(5,3))
    rated_df[rated_df['Rating']==rating]['Total Discount'].hist(bins=20)
    plt.title(f"Rating {rating}")
    plt.xlabel("Total Discount")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# Null Hypothesis (H₀):
#  There is no significant difference in the distribution of total discount across the different customer
#  rating categories.

# Alternative Hypothesis (H₁):
#  There is a significant difference in the distribution of total discount across at least one customer
#  rating category.


In [ ]:
# now we are performing kruskal wallis H test for statistical testing

In [ ]:
from scipy.stats import kruskal

group1 = rated_df[rated_df["Rating"] == 1]["Total Discount"]
group2 = rated_df[rated_df["Rating"] == 2]["Total Discount"]
group3 = rated_df[rated_df["Rating"] == 3]["Total Discount"]
group4 = rated_df[rated_df["Rating"] == 4]["Total Discount"]
group5 = rated_df[rated_df["Rating"] == 5]["Total Discount"]

stat, p = kruskal(group1, group2, group3, group4, group5)

print(f"Kruskal-Wallis H Statistic: {stat:.4f}")
print(f"P-value: {p:.4f}")

In [ ]:
# calculating effect size
N = len(rated_df)      # Total number of observations
k = 5                  # Number of rating groups

epsilon_sq = (stat - k + 1) / (N - k)

print(f"Epsilon-squared (ε²): {epsilon_sq:.4f}")

In [ ]:
# No statistically significant association was found between the total discount applied to an order and
# customer ratings. Although slight differences in discount amounts were observed across the rating categories
# during exploratory data analysis, these differences were not statistically significant. Additionally, the
# negligible effect size indicates a minimal practical relationship between the two variables.

### Does rider wait time differ across different order ready marking statuses?

In [ ]:
temp=df.dropna(subset=['Rider wait time (minutes)'])

In [ ]:
summary = temp.groupby('Order Ready Marked')['Rider wait time (minutes)'].agg(
    Count='count',
    Mean='mean',
    Median='median',
    Std='std',
    Min='min',
    Max='max'
).round(2)

summary

In [ ]:
# performing shapiro wilk test to check whether rider wait time follows normal distribution within
# each order ready marked category
from scipy.stats import shapiro

for status, group in temp.groupby('Order Ready Marked'):
    stat, p = shapiro(group['Rider wait time (minutes)'])
    print(f"{status}")
    print(f"Shapiro-Wilk Statistic = {stat:.4f}")
    print(f"p-value = {p:.4f}\n")

In [ ]:
# statistical testing using kruskal wallis h test
from scipy.stats import kruskal

correct = temp[temp['Order Ready Marked'] == 'Correctly']['Rider wait time (minutes)']
incorrect = temp[temp['Order Ready Marked'] == 'Incorrectly']['Rider wait time (minutes)']
missed = temp[temp['Order Ready Marked'] == 'Missed']['Rider wait time (minutes)']

stat, p = kruskal(correct, incorrect, missed)

print(f"Kruskal-Wallis H Statistic = {stat:.4f}")
print(f"p-value = {p:.4f}")

In [ ]:
# checking effect size
n = len(temp)
k = temp['Order Ready Marked'].nunique()
H = 1465.7971

epsilon_squared = (H - k + 1) / (n - k)
print(f"Epsilon Squared = {epsilon_squared:.4f}")

### Does kitchen preparation time differ across different times of the day?

In [ ]:
temp=df.dropna(subset=['KPT duration (minutes)'])

In [ ]:
summary = temp.groupby('Time_of_Day')['KPT duration (minutes)'].agg(
    Count='count',
    Mean='mean',
    Median='median',
    Std='std'
).round(2)

summary

In [ ]:
temp['KPT duration (minutes)'].skew()

In [ ]:
# applying kruskal wallis H test
from scipy.stats import kruskal

# Create groups
evening = temp[temp['Time_of_Day'] == 'Evening/Dinner']['KPT duration (minutes)']
lunch = temp[temp['Time_of_Day'] == 'Lunch/Afternoon']['KPT duration (minutes)']
late_night = temp[temp['Time_of_Day'] == 'Late Night']['KPT duration (minutes)']

# Kruskal-Wallis test
H, p = kruskal(evening, lunch, late_night)

print(f"H-statistic: {H:.4f}")
print(f"p-value: {p:.4f}")

In [ ]:
# Number of groups
k = 3

# Total sample size
n = len(temp)

# Epsilon squared
epsilon_squared = (H - k + 1) / (n - k)

print(f"Epsilon Squared: {epsilon_squared:.4f}")

In [ ]:
# Kitchen preparation time differed statistically across different times of the day. However, the negligible effect size indicates that
# time of day has minimal practical influence on kitchen preparation time, suggesting that preparation times remain relatively consistent
# across different periods of the day.

### Is there an association between rider wait time and customer ratings?

In [ ]:
temp=df[(df['Rating'].isnull()==False) & (df['Rider wait time (minutes)'].isnull()==False)].copy()

In [ ]:
summary = (
    temp.groupby('Rating')['Rider wait time (minutes)']
    .agg(
        Count='count',
        Mean='mean',
        Median='median',
        Std='std'
    )
    .round(2)
)

summary

In [ ]:
temp['Rider wait time (minutes)'].skew()

In [ ]:
# checking normality assumption using shapiro wilk test
from scipy.stats import shapiro

for rating in sorted(temp['Rating'].unique()):
    stat, p = shapiro(
        temp.loc[temp['Rating'] == rating, 'Rider wait time (minutes)']
    )
    print(f"Rating {rating:.0f}: W = {stat:.4f}, p = {p:.4f}")

In [ ]:
# statistical testing of hypothesis using kruskal wallis test
from scipy.stats import kruskal

groups = [
    group['Rider wait time (minutes)'].values
    for _, group in temp.groupby('Rating')
]

H, p = kruskal(*groups)

print(f"H-statistic: {H:.4f}")
print(f"p-value: {p:.4f}")

In [ ]:
k = temp['Rating'].nunique()   # Number of groups
n = len(temp)                  # Total observations

epsilon_squared = (H - k + 1) / (n - k)

print(f"Epsilon Squared: {epsilon_squared:.4f}")

In [ ]:
# The analysis found no statistically significant association between rider wait time and customer ratings. This suggests that differences
#  in rider wait time are not reflected in customer ratings within this dataset, indicating that rider wait time alone is unlikely to be a
#  meaningful factor associated with customers' rating behavior.

### 3.7 Demand by Day, Time and Month

### Does customer order demand vary across different days of the week?

In [ ]:
df['Order Date']=pd.to_datetime(df['Order Date'])

In [ ]:
temp=df['Order Date'].dt.day_name().value_counts()

In [ ]:
# H₀ (Null Hypothesis):
# Customer order demand is equally distributed across all days of the week.

# H₁ (Alternative Hypothesis):
# Customer order demand varies across at least one day of the week.

In [ ]:
from scipy.stats import chisquare

# Observed frequencies
observed = temp.values

# Expected frequencies under H0 (equal distribution)
expected = [observed.sum() / len(observed)] * len(observed)

# Chi-square Goodness-of-Fit Test
chi2_stat, p_value = chisquare(f_obs=observed,
                               f_exp=expected)

print(f"Chi-square Statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
import numpy as np

observed = np.array(observed)
expected = np.array(expected)

cohens_w = np.sqrt(np.sum((observed - expected)**2 / expected) / observed.sum())

print(f"Cohen's w: {cohens_w:.4f}")

In [ ]:
# Customer order demand varied significantly across the days of the week. The statistical analysis confirmed that the observed
# differences in order volume were unlikely to have occurred due to random variation alone. However, the small effect size suggests that
# while weekday demand patterns exist, the overall magnitude of variation across the week is relatively modest

### Does customer order demand vary across different times of the day?

In [ ]:
# Order demand appeared to vary considerably across different times of the day. The Evening/Dinner period recorded the highest number
# of orders (12,463), accounting for more than half of all orders in the dataset. Lunch/Afternoon followed with 6,186 orders, while
# Late Night had the lowest demand with 2,672 orders. These descriptive patterns suggest that customer ordering behavior differs across
# time periods; however, statistical testing is required to determine whether the observed differences are statistically significant
# rather than due to random variation.

In [ ]:
# Null Hypothesis (H₀):
#  Customer orders are equally distributed across the different times of the day.

# Alternative Hypothesis (H₁):
#  Customer orders are not equally distributed across the different times of the day.


In [ ]:
# performing chi square goodness of fit test
from scipy.stats import chisquare

# Observed frequencies
observed = df['Time_of_Day'].value_counts().sort_index().values

# Expected frequencies (equal distribution)
expected = [observed.sum() / len(observed)] * len(observed)

# Chi-square test
chi2, p = chisquare(f_obs=observed, f_exp=expected)

print(f"Chi-square Statistic: {chi2:.4f}")
print(f"P-value: {p:.4f}")

In [ ]:
# checking effect size
import numpy as np

cohens_w = np.sqrt(
    np.sum((observed - expected) ** 2 / expected) / observed.sum()
)

print(f"Cohen's w: {cohens_w:.4f}")

In [ ]:
# Investigation Conclusion :-

# Customer order demand varied significantly across different times of the day. The statistical analysis confirmed that the observed
# differences in order volume were unlikely to have occurred due to random variation alone. Furthermore, the large effect size indicates
# that time of day is a strong factor associated with customer order demand in the dataset. Detailed demand patterns across individual
# time periods are discussed in the Driver Analysis section.

### Does customer order demand vary across different months?

In [ ]:
df['month_name']=df['Order Date'].dt.month_name()

In [ ]:
from scipy.stats import chisquare

# Observed frequencies
observed = df['month_name'].value_counts().sort_index().values

# Expected frequencies (equal distribution)
expected = [observed.sum() / len(observed)] * len(observed)

# Chi-square Goodness-of-Fit Test
chi2, p = chisquare(f_obs=observed, f_exp=expected)

print(f"Chi-square Statistic: {chi2:.4f}")
print(f"P-value: {p:.4f}")

In [ ]:
import numpy as np

# Cohen's w
cohens_w = np.sqrt(
    np.sum((observed - expected) ** 2 / expected) / observed.sum()
)

print(f"Cohen's w: {cohens_w:.4f}")

### 3.8 Discount Patterns

### Does the total discount provided per order vary across different times of the day?

In [ ]:
temp=df.groupby('Time_of_Day')['Total Discount'].agg(Count='count',
        Mean_discount='mean',
        Median_discount='median',
        Std='std').round(2)

temp

In [ ]:
df['Total Discount'].skew()

In [ ]:
# Total Discount is highly right skewed

In [ ]:
from scipy.stats import kruskal

# Create groups
evening = df.loc[df['Time_of_Day'] == 'Evening/Dinner', 'Total Discount'].dropna()
lunch = df.loc[df['Time_of_Day'] == 'Lunch/Afternoon', 'Total Discount'].dropna()
late_night = df.loc[df['Time_of_Day'] == 'Late Night', 'Total Discount'].dropna()

# Kruskal-Wallis Test
H_stat, p_value = kruskal(evening, lunch, late_night)

print(f"H-statistic: {H_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
N = len(evening) + len(lunch) + len(late_night)
k = 3

epsilon_squared = (H_stat - k + 1) / (N - k)

print(f"Epsilon Squared: {epsilon_squared:.4f}")

In [ ]:
# The analysis detected statistically significant differences in total discount across different times of
# the day. However, the negligible effect size (ε² = 0.0077) indicates that these differences were very
# small in magnitude, suggesting that discount levels remained largely consistent across the different
# time periods from a practical business perspective.

In [ ]:
import scikit_posthocs as sp

dunn_discount = sp.posthoc_dunn(
    df,
    val_col='Total Discount',
    group_col='Time_of_Day',
    p_adjust='holm'
)

dunn_discount

### Does the total discount provided per order vary across different days of the week?

In [ ]:
df['day_of_week']=df['Order Date'].dt.day_name()

In [ ]:
temp=df.groupby('day_of_week')['Total Discount'].agg(Count='count',
        Mean_discount='mean',
        Median_discount='median',
        Std='std').round(2)

temp

In [ ]:
from scipy.stats import kruskal

# Create groups
groups = [
    group['Total Discount'].dropna()
    for _, group in df.groupby('day_of_week')
]

# Perform Kruskal-Wallis test
H_stat, p_value = kruskal(*groups)

print(f"H-statistic: {H_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
N = df['Total Discount'].notna().sum()
k = df['day_of_week'].nunique()

epsilon_squared = (H_stat - k + 1) / (N - k)

print(f"Epsilon Squared: {epsilon_squared:.4f}")

In [ ]:
import scikit_posthocs as sp

dunn_day_discount = sp.posthoc_dunn(
    df,
    val_col='Total Discount',
    group_col='day_of_week',
    p_adjust='holm'
)

dunn_day_discount.round(4)

In [ ]:
# The analysis detected statistically significant differences in total discount across different days of
# the week. However, the negligible effect size (ε² = 0.0095) indicates that these differences were very
# small in magnitude, suggesting that discount levels remained largely consistent across the different
# weekdays from a practical business perspective.

### 3.9 Weekly Cohort Retention

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

df = df.sort_values('Order Date').reset_index(drop=True)



In [ ]:
start_date = df['Order Date'].min()

df['Week_Number'] = ((df['Order Date'] - start_date).dt.days // 7) + 1

In [ ]:
first_week = (
    df.groupby('Customer ID')['Week_Number']
      .min()
      .reset_index()
      .rename(columns={'Week_Number':'Cohort_Week'})
)

In [ ]:
df = df.merge(first_week, on='Customer ID', how='left')

In [ ]:
df['Week_Index'] = df['Week_Number'] - df['Cohort_Week']

In [ ]:
# One record per customer per cohort-week
cohort_data = (
    df[['Customer ID', 'Cohort_Week', 'Week_Index']]
    .drop_duplicates(
        subset=['Customer ID', 'Cohort_Week', 'Week_Index']
    )
)

In [ ]:
cohort_counts = (
    cohort_data
    .groupby(['Cohort_Week', 'Week_Index'])['Customer ID']
    .nunique()
    .unstack(fill_value=0)
)

In [ ]:
cohort_retention = (
    cohort_counts
    .div(cohort_counts[0], axis=0)
    .mul(100)
    .round(2)
)

cohort_retention

In [ ]:
# Maximum retention window
max_retention_weeks = 10

# Total number of weeks in the dataset
total_weeks = df['Week_Number'].max()

# Latest cohort that has a complete 10-week follow-up
last_valid_cohort = total_weeks - max_retention_weeks

print(last_valid_cohort)

cohort_retention_10w = cohort_retention.loc[
    cohort_retention.index <= last_valid_cohort,
    :max_retention_weeks
]

cohort_retention_10w

In [ ]:
# average retention by week
# Average retention across all cohorts
average_retention = (
    cohort_retention_10w
    .mean(axis=0)
    .round(2)
    .reset_index()
)

average_retention.columns = ['Weeks_Since_First_Order', 'Average_Retention (%)']

average_retention

In [ ]:
# On average, 8.90% of newly acquired customers returned to place another order one week after their first purchase. The retention rate
# gradually declined over subsequent weeks, reaching 4.66% by the tenth week after acquisition. This indicates that customer retention
# decreases over time, with relatively few customers continuing to place orders several weeks after joining the platform.

### 3.10 Customer Ordering Behaviour

### Do customer retention patterns differ across weekly acquisition cohorts?

In [ ]:
cohort_avg_retention = (
    cohort_retention_10w.iloc[:, 1:]   # Exclude Week_Index = 0
    .mean(axis=1)
    .round(2)
)

cohort_avg_retention

In [ ]:
from scipy.stats import spearmanr

corr, p_value = spearmanr(
    cohort_avg_retention.index,
    cohort_avg_retention.values
)

print(corr, p_value)

### How frequently do customers place orders on the platform?

In [ ]:
order_frequency['Percentage_of_Customers'] = (
    order_frequency['Number_of_Customers']
    / order_frequency['Number_of_Customers'].sum()
    * 100
).round(2)

order_frequency

In [ ]:
customer_spending = (
    df.groupby('Customer ID')
      .agg(
          Total_Orders=('Order ID', 'count'),
          Total_Spend=('Total', 'sum'),
          Total_Discount=('Total Discount', 'sum')
      )
      .reset_index()
)

customer_spending.head()

### 3.11 Customer Value & Revenue Concentration

### Do a small group of high-spending customers contribute a disproportionately large share of total revenue during the study period?

In [ ]:
# Sort customers by spending
customer_spending_sorted = (
    customer_spending
    .sort_values('Total_Spend', ascending=False)
    .reset_index(drop=True)
)

# Cumulative revenue
customer_spending_sorted['Cumulative_Revenue'] = (
    customer_spending_sorted['Total_Spend'].cumsum()
)

# Revenue share (%)
customer_spending_sorted['Cumulative_Revenue_%'] = (
    customer_spending_sorted['Cumulative_Revenue']
    / customer_spending_sorted['Total_Spend'].sum()
    * 100
)

# Customer share (%)
customer_spending_sorted['Customer_%'] = (
    (customer_spending_sorted.index + 1)
    / len(customer_spending_sorted)
    * 100
)

customer_spending_sorted.head()

In [ ]:
for pct in [10, 20, 30, 40, 50]:
    revenue_share = customer_spending_sorted.loc[
        customer_spending_sorted['Customer_%'] <= pct,
        'Total_Spend'
    ].sum() / customer_spending_sorted['Total_Spend'].sum() * 100

    print(f"Top {pct}% customers contribute {revenue_share:.2f}% of total revenue")

### Is higher customer spending primarily driven by more frequent ordering or by higher-value orders?

In [ ]:
customer_spending['Average_Order_Value'] = (
    customer_spending['Total_Spend'] /
    customer_spending['Total_Orders']
)

In [ ]:
# Sort customers by total spend
customer_spending_sorted = customer_spending.sort_values(
    by='Total_Spend',
    ascending=False
).reset_index(drop=True)

# Number of customers in top 20%
top_n = int(len(customer_spending_sorted) * 0.20)

# Create customer segment
customer_spending_sorted['Customer_Segment'] = 'Remaining 80%'
customer_spending_sorted.loc[:top_n-1, 'Customer_Segment'] = 'Top 20%'

In [ ]:
comparison = (
    customer_spending_sorted
    .groupby('Customer_Segment')
    .agg(
        Customers=('Customer ID', 'count'),
        Mean_Orders=('Total_Orders', 'mean'),
        Median_Orders=('Total_Orders', 'median'),
        Mean_AOV=('Average_Order_Value', 'mean'),
        Median_AOV=('Average_Order_Value', 'median'),
        Mean_Total_Spend=('Total_Spend', 'mean'),
        Median_Total_Spend=('Total_Spend', 'median')
    )
    .round(2)
)

comparison

### 3.12 Reorder Behaviour

### Is higher customer spending associated with shorter reorder intervals?

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

df_sorted = df.sort_values(['Customer ID', 'Order Date'])


In [ ]:
df_sorted['Days_Between_Orders'] = (
    df_sorted.groupby('Customer ID')['Order Date']
             .diff()
             .dt.days
)


customer_reorder = (
    df_sorted.groupby('Customer ID')
             .agg(
                 Avg_Reorder_Days=('Days_Between_Orders', 'mean'),
                 Median_Reorder_Days=('Days_Between_Orders', 'median')
             )
             .reset_index()
)

In [ ]:
customer_analysis = customer_spending_sorted.merge(
    customer_reorder,
    on='Customer ID',
    how='left'
)

In [ ]:
comparison = (
    customer_analysis
    .groupby('Customer_Segment')
    .agg(
        Customers=('Customer ID', 'count'),
        Mean_Reorder_Days=('Avg_Reorder_Days', 'mean'),
        Median_Reorder_Days=('Median_Reorder_Days', 'median')
    )
    .round(2)
)

comparison

In [ ]:
order_distribution = (
    customer_orders['Number_of_Orders']
    .value_counts()
    .sort_index()
    .reset_index()
)

order_distribution.columns = ['Number_of_Orders', 'Number_of_Customers']

order_distribution

In [ ]:
order_distribution['Percentage_of_Customers'] = (
    order_distribution['Number_of_Customers']
    / order_distribution['Number_of_Customers'].sum()
    * 100
).round(2)

order_distribution

### 3.13 Discounts and Customer Frequency

### Do customers with different purchasing frequencies receive different discount levels per order?

In [ ]:
customer_orders['Customer_Frequency_Segment'] = np.where(
    customer_orders['Number_of_Orders'] == 1,
    'One-Time Customer',
    np.where(
        customer_orders['Number_of_Orders'].between(2, 3),
        'Occasional Customer',
        'Frequent Customer'
    )
)

customer_orders.head()

In [ ]:
df_segmented = df.merge(
    customer_orders[['Customer ID', 'Customer_Frequency_Segment']],
    on='Customer ID',
    how='left'
)

df_segmented.head()

In [ ]:
discount_summary = (
    df_segmented
    .groupby('Customer_Frequency_Segment')
    .agg(
        Number_of_Orders=('Order ID', 'count'),
        Mean_Discount=('Total Discount', 'mean'),
        Median_Discount=('Total Discount', 'median'),
        Std_Discount=('Total Discount', 'std')
    )
    .round(2)
)

discount_summary

In [ ]:
from scipy.stats import kruskal

one_time = df_segmented.loc[
    df_segmented['Customer_Frequency_Segment'] == 'One-Time Customer',
    'Total Discount'
]

occasional = df_segmented.loc[
    df_segmented['Customer_Frequency_Segment'] == 'Occasional Customer',
    'Total Discount'
]

frequent = df_segmented.loc[
    df_segmented['Customer_Frequency_Segment'] == 'Frequent Customer',
    'Total Discount'
]

H_stat, p_value = kruskal(one_time, occasional, frequent)

print(f"H-statistic: {H_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
N = len(df_segmented)
k = 3

epsilon_squared = (H_stat - k + 1) / (N - k)

print(f"Epsilon Squared: {epsilon_squared:.4f}")

In [ ]:
import scikit_posthocs as sp

dunn = sp.posthoc_dunn(
    df_segmented,
    val_col='Total Discount',
    group_col='Customer_Frequency_Segment',
    p_adjust='bonferroni'
)

dunn

### 3.14 Cancellation Behaviour by Customer Type

### Is order cancellation behavior different between one-time and repeat customers?

In [ ]:
customer_orders['Customer_Type'] = customer_orders[
    'Customer_Frequency_Segment'
].replace({
    'One-Time Customer': 'One-Time Customer',
    'Occasional Customer': 'Repeat Customer',
    'Frequent Customer': 'Repeat Customer'
})

In [ ]:
df_cancel = df.merge(
    customer_orders[['Customer ID', 'Customer_Type']],
    on='Customer ID',
    how='left'
)

In [ ]:
cancellation_summary = (
    df_cancel
    .groupby('Customer_Type')
    .agg(
        Total_Orders=('Order ID', 'count'),
        Cancelled_Orders=('Customer Cancelled',
                          lambda x: (x == 'Yes').sum())
    )
)

cancellation_summary['Cancellation_Rate (%)'] = (
    cancellation_summary['Cancelled_Orders']
    / cancellation_summary['Total_Orders']
    * 100
).round(2)

cancellation_summary

In [ ]:
contingency_table = pd.crosstab(
    df_cancel['Customer_Type'],
    df_cancel['Customer Cancelled']
)

contingency_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of Freedom: {dof}")

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

expected_df

In [ ]:
import numpy as np

n = contingency_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Cramer's V: {cramers_v:.4f}")

### 3.15 Revenue Contribution by Customer Type

### Is the platform's revenue primarily driven by one-time customers or repeat customers?

In [ ]:
df_revenue = df.merge(
    customer_orders[['Customer ID', 'Customer_Type']],
    on='Customer ID',
    how='left'
)


revenue_summary = (
    df_revenue
    .groupby('Customer_Type')
    .agg(
        Number_of_Customers=('Customer ID', 'nunique'),
        Total_Orders=('Order ID', 'count'),
        Total_Revenue=('Total', 'sum')
    )
)

revenue_summary['Revenue_Contribution (%)'] = (
    revenue_summary['Total_Revenue']
    / revenue_summary['Total_Revenue'].sum()
    * 100
).round(2)

revenue_summary['Average_Revenue_per_Customer'] = (
    revenue_summary['Total_Revenue']
    / revenue_summary['Number_of_Customers']
).round(2)

revenue_summary

In [ ]:
customer_spending = customer_spending.sort_values(
    by='Total_Spend',
    ascending=False
).reset_index(drop=True)


n_customers = len(customer_spending)

top_20_count = int(np.ceil(n_customers * 0.20))

customer_spending['Customer_Segment'] = 'Remaining 80%'

customer_spending.loc[
    :top_20_count-1,
    'Customer_Segment'
] = 'Top 20%'


df_revenue = df.merge(
    customer_spending[['Customer ID', 'Customer_Segment']],
    on='Customer ID',
    how='left'
)


rating_summary = (
    df_revenue
    .groupby('Customer_Segment')
    .agg(
        Total_Orders=('Order ID', 'count'),
        Rated_Orders=('Rating', lambda x: x.notna().sum())
    )
)

rating_summary['Rating_Coverage (%)'] = (
    rating_summary['Rated_Orders']
    / rating_summary['Total_Orders']
    * 100
).round(2)

rating_summary

In [ ]:
rating_analysis = (
    df_revenue
    .dropna(subset=['Rating'])
    .groupby('Customer_Segment')
    .agg(
        Rated_Orders=('Rating', 'count'),
        Mean_Rating=('Rating', 'mean'),
        Median_Rating=('Rating', 'median'),
        Std_Rating=('Rating', 'std')
    )
    .round(2)
)

rating_analysis

### 3.16 Restaurant-Level Analysis

### 3.16 Restaurant-Level Business Summary

In [ ]:
# Restaurant Business Performance Summary

restaurant_summary = (
    df.groupby('Restaurant name')
      .agg(
          Unique_Customers=('Customer ID', 'nunique'),
          Total_Orders=('Order ID', 'count'),
          Total_Revenue=('Total', 'sum'),
          Customer_Cancelled_Orders=('Customer Cancelled',
                                     lambda x: (x == 'Yes').sum()),
          Zomato_Cancelled_Orders=('Zomato_Cancelled',
                                   lambda x: (x == 'Yes').sum())
      )
      .reset_index()
)

# ----------------------------
# Order Share
# ----------------------------

restaurant_summary['Order_Share (%)'] = (
    restaurant_summary['Total_Orders']
    / restaurant_summary['Total_Orders'].sum()
    * 100
)

# ----------------------------
# Revenue Share
# ----------------------------

restaurant_summary['Revenue_Share (%)'] = (
    restaurant_summary['Total_Revenue']
    / restaurant_summary['Total_Revenue'].sum()
    * 100
)

# ----------------------------
# Repeat Customer Rate
# ----------------------------

restaurant_customer = (
    df.groupby(['Restaurant name', 'Customer ID'])
      .size()
      .reset_index(name='Orders')
)

repeat_rate = (
    restaurant_customer
    .groupby('Restaurant name')
    .apply(lambda x: (x['Orders'] >= 2).mean() * 100)
    .reset_index(name='Repeat_Customer_Rate (%)')
)

restaurant_summary = restaurant_summary.merge(
    repeat_rate,
    on='Restaurant name',
    how='left'
)

# ----------------------------
# Cancellation Rates
# ----------------------------

restaurant_summary['Customer_Cancellation_Rate (%)'] = (
    restaurant_summary['Customer_Cancelled_Orders']
    / restaurant_summary['Total_Orders']
    * 100
)

restaurant_summary['Zomato_Cancellation_Rate (%)'] = (
    restaurant_summary['Zomato_Cancelled_Orders']
    / restaurant_summary['Total_Orders']
    * 100
)

# ----------------------------
# Final Table
# ----------------------------

restaurant_summary = (
    restaurant_summary[
        [
            'Restaurant name',
            'Unique_Customers',
            'Total_Orders',
            'Order_Share (%)',
            'Repeat_Customer_Rate (%)',
            'Total_Revenue',
            'Revenue_Share (%)',
            'Customer_Cancellation_Rate (%)',
            'Zomato_Cancellation_Rate (%)'
        ]
    ]
    .round({
        'Order_Share (%)': 2,
        'Repeat_Customer_Rate (%)': 2,
        'Total_Revenue': 2,
        'Revenue_Share (%)': 2,
        'Customer_Cancellation_Rate (%)': 2,
        'Zomato_Cancellation_Rate (%)': 2
    })
    .sort_values(by='Total_Revenue', ascending=False)
    .reset_index(drop=True)
)

restaurant_summary

### How do operational efficiency and pricing characteristics differ across restaurant brands during the study period?

In [ ]:
restaurant_operational = (
    df.groupby('Restaurant name')
      .agg(
          Median_KPT_Minutes=('KPT duration (minutes)', 'median'),
          Median_Rider_Wait_Minutes=('Rider wait time (minutes)', 'median'),
          Median_Packaging_Charges=('Packaging charges', 'median'),
          Median_Total_Discount=('Total Discount', 'median'),
          Median_Average_Order_Value=('Total', 'median')
      )
      .round(2)
      .reset_index()
)

restaurant_operational

### 3.17 Distance & Customer Cancellation

In [ ]:
def distance_group(x):
    if x <= 2:
        return 'Nearby (≤2 km)'
    elif x <= 5:
        return 'Short (3–5 km)'
    elif x <= 8:
        return 'Medium (6–8 km)'
    else:
        return 'Long (≥9 km)'

df['Distance_Group'] = df['Distance'].apply(distance_group)

df['Distance_Group'].value_counts()

In [ ]:
distance_summary = (
    df.groupby('Distance_Group')
      .agg(
          Total_Orders=('Order ID', 'count'),
          Customer_Cancelled=('Customer Cancelled',
                              lambda x: (x == 'Yes').sum())
      )
      .reset_index()
)

distance_summary['Customer_Cancellation_Rate (%)'] = (
    distance_summary['Customer_Cancelled']
    / distance_summary['Total_Orders']
    * 100
).round(2)

distance_summary

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

contingency_table = pd.crosstab(
    df['Distance_Group'],
    df['Customer Cancelled']
)

contingency_table

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of Freedom: {dof}")

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

expected_df

In [ ]:
print("Expected frequencies less than 5:")
print((expected_df < 5).sum().sum())

print("\nMinimum expected frequency:")
print(expected_df.min().min())

In [ ]:
n = contingency_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Cramer's V: {cramers_v:.4f}")

### 3.18 Monthly Cancellation Trends

### How did customer-initiated and platform-initiated order cancellation rates vary over the study period?

In [ ]:
monthly_cancellation = (
    df.groupby('month_name')
      .agg(
          Total_Orders=('Order ID', 'count'),
          Customer_Cancelled=('Customer Cancelled', lambda x: (x == 'Yes').sum()),
          Zomato_Cancelled=('Zomato_Cancelled', lambda x: (x == 'Yes').sum())
      )
      .reset_index()
)




In [ ]:
monthly_cancellation['Customer_Cancellation_Rate (%)'] = (
    monthly_cancellation['Customer_Cancelled']
    / monthly_cancellation['Total_Orders']
    * 100
).round(2)

monthly_cancellation['Zomato_Cancellation_Rate (%)'] = (
    monthly_cancellation['Zomato_Cancelled']
    / monthly_cancellation['Total_Orders']
    * 100
).round(2)

monthly_cancellation['Overall_Cancellation_Rate (%)'] = (
    (monthly_cancellation['Customer_Cancelled'] +
     monthly_cancellation['Zomato_Cancelled'])
    / monthly_cancellation['Total_Orders']
    * 100
).round(2)

In [ ]:
month_order = [
    'September',
    'October',
    'November',
    'December',
    'January'
]

monthly_cancellation['month_name'] = pd.Categorical(
    monthly_cancellation['month_name'],
    categories=month_order,
    ordered=True
)

monthly_cancellation = monthly_cancellation.sort_values('month_name')

### 3.19 Restaurant Demand, Order Value & Discounts by Time of Day

### How do customer demand, order value, and promotional discounts vary across different times of the day for each restaurant?

In [ ]:
# Order counts
counts = pd.crosstab(
    df['Restaurant name'],
    df['Time_of_Day']
)

# Percentage distribution
percentages = (
    pd.crosstab(
        df['Restaurant name'],
        df['Time_of_Day'],
        normalize='index'
    ) * 100
).round(2)

# Create final table
restaurant_time_summary = counts.copy()

for col in percentages.columns:
    restaurant_time_summary[f'{col} (%)'] = percentages[col]

restaurant_time_summary = restaurant_time_summary.reset_index()

restaurant_time_summary

In [ ]:
restaurant_time_orders = (
    pd.crosstab(
        df['Restaurant name'],
        df['Time_of_Day']
    )
)

restaurant_time_percent = (
    pd.crosstab(
        df['Restaurant name'],
        df['Time_of_Day'],
        normalize='index'
    ) * 100
).round(2)

restaurant_demand = restaurant_time_orders.copy()

for col in restaurant_time_percent.columns:
    restaurant_demand[f'{col} (%)'] = restaurant_time_percent[col]

restaurant_demand = restaurant_demand.reset_index()

restaurant_demand

In [ ]:
restaurant_order_value = (
    df.pivot_table(
        index='Restaurant name',
        columns='Time_of_Day',
        values='Total',
        aggfunc='median'
    )
    .round(2)
    .reset_index()
)

restaurant_order_value

In [ ]:
restaurant_discount = (
    df.pivot_table(
        index='Restaurant name',
        columns='Time_of_Day',
        values='Total Discount',
        aggfunc='median'
    )
    .round(2)
    .reset_index()
)

restaurant_discount

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Contingency table
distance_cancel = pd.crosstab(
    df['Distance_Group'],
    df['Customer Cancelled']
)

# Chi-square test
chi2, p, dof, expected = chi2_contingency(distance_cancel)

# Pearson standardized residuals
residuals = (
    distance_cancel - expected
) / np.sqrt(expected)

print("Standardized Residuals:")
display(
    pd.DataFrame(
        residuals,
        index=distance_cancel.index,
        columns=distance_cancel.columns
    ).round(2)
)

## 4. Operational Driver Analysis

### Does kitchen preparation time differ across order-ready marking statuses?

In [ ]:
# Keep only rows where KPT and Order Ready Marked are available
kpt_ready = df[
    df['KPT duration (minutes)'].notna() &
    df['Order Ready Marked'].notna()
].copy()

In [ ]:
kpt_ready.groupby('Order Ready Marked')['KPT duration (minutes)'].agg(
    Count='count',
    Mean='mean',
    Median='median',
    Std='std'
).round(2)

In [ ]:
from scipy.stats import kruskal

groups = [
    group['KPT duration (minutes)'].values
    for _, group in kpt_ready.groupby('Order Ready Marked')
]

stat, p = kruskal(*groups)

print("Kruskal-Wallis H:", round(stat, 4))
print("P-value:", round(p, 4))

In [ ]:
n = len(kpt_ready)
k = kpt_ready['Order Ready Marked'].nunique()

epsilon_squared = (stat - k + 1) / (n - k)

print("Epsilon-squared:", round(epsilon_squared, 4))

In [ ]:
import scikit_posthocs as sp

dunn_result = sp.posthoc_dunn(
    kpt_ready,
    val_col='KPT duration (minutes)',
    group_col='Order Ready Marked',
    p_adjust='holm'
)

dunn_result

### Is order-ready marking status associated with time of day?

In [ ]:
ready_time = df[
    df['Time_of_Day'].notna() &
    df['Order Ready Marked'].notna()
].copy()

In [ ]:
pd.crosstab(
    ready_time['Time_of_Day'],
    ready_time['Order Ready Marked']
)

time_ready_pct = pd.crosstab(
    ready_time['Time_of_Day'],
    ready_time['Order Ready Marked'],
    normalize='index'
) * 100

time_ready_pct.round(2)

In [ ]:


pct_table = (
    pd.crosstab(
        ready_time['Time_of_Day'],
        ready_time['Order Ready Marked'],
        normalize='index'
    ) * 100
).round(2)

pct_table

In [ ]:
from scipy.stats import chi2_contingency

# Contingency table
contingency_table = pd.crosstab(
    ready_time['Time_of_Day'],
    ready_time['Order Ready Marked']
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-Square Statistic:", round(chi2, 4))
print("P-value:", round(p, 4))
print("Degrees of Freedom:", dof)

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

print("Expected Frequencies:")
display(expected_df.round(2))

print("Minimum Expected Frequency:",
      expected_df.min().min())

print("Expected frequencies < 5:",
      (expected_df < 5).sum().sum())

In [ ]:
import numpy as np

n = contingency_table.to_numpy().sum()
r, k = contingency_table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

print("Cramer's V:", round(cramers_v, 4))

## 5. Cohort & Retention Analysis

In [ ]:
# Make sure order date is datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

# First order date for each customer
first_order = df.groupby('Customer ID')['Order Date'].min().rename('First_Order_Date')

# Add first order date to every order
df_cohort = df.merge(first_order, on='Customer ID', how='left')

# Acquisition month
df_cohort['Cohort_Month'] = df_cohort['First_Order_Date'].dt.to_period('M')

# Days since customer's first order
df_cohort['Days_Since_Acquisition'] = (
    df_cohort['Order Date'] - df_cohort['First_Order_Date']
).dt.days

# Check cohort sizes
cohort_check = (
    df_cohort.groupby('Cohort_Month')['Customer ID']
    .nunique()
    .reset_index(name='Customers')
)

cohort_check

In [ ]:
study_end = df_cohort['Order Date'].max()

cohort_check['30_Day_Window_Available'] = (
    cohort_check['Cohort_Month'].dt.to_timestamp() + pd.Timedelta(days=30)
    <= study_end
)

cohort_check

In [ ]:
customer_check = (
    df_cohort.groupby('Customer ID')
    .agg(
        First_Order_Date=('First_Order_Date', 'first'),
        Last_Observed_Date=('Order Date', 'max')
    )
    .reset_index()
)

customer_check['Complete_30_Days'] = (
    customer_check['First_Order_Date'] + pd.Timedelta(days=30)
    <= study_end
)

customer_check['Cohort_Month'] = (
    customer_check['First_Order_Date'].dt.to_period('M')
)

customer_check.groupby('Cohort_Month')['Complete_30_Days'].agg(
    Customers='count',
    Complete_30_Days='sum'
).reset_index()

In [ ]:
# Keep only customers with a complete 30-day observation window
complete_customers = customer_check.loc[
    customer_check['Complete_30_Days'],
    'Customer ID'
]

df_30 = df_cohort[
    df_cohort['Customer ID'].isin(complete_customers) &
    (df_cohort['Days_Since_Acquisition'] <= 30)
].copy()

# Keep September–December cohorts only
df_30 = df_30[
    df_30['Cohort_Month'].isin(
        pd.period_range('2024-09', '2024-12', freq='M')
    )
].copy()

In [ ]:
cohort_cancellation_check = (
    df_30.groupby('Cohort_Month')
    .agg(
        Customers=('Customer ID', 'nunique'),
        Total_Orders=('Customer ID', 'size'),
        Customer_Cancelled=('Customer Cancelled', lambda x: (x == 'Yes').sum())
    )

    .reset_index()
)

cohort_cancellation_check['Customer_Cancellation_Rate (%)'] = (
    cohort_cancellation_check['Customer_Cancelled']
    / cohort_cancellation_check['Total_Orders']
    * 100
).round(2)

cohort_cancellation_check

In [ ]:
customer_cohort_cancel = (
    df_30.groupby(['Cohort_Month', 'Customer ID'])
    .agg(
        Cancelled_Order=('Customer Cancelled',
                         lambda x: (x == 'Yes').sum())
    )
    .reset_index()
)

customer_cohort_cancel['Cancelled_At_Least_Once'] = (
    customer_cohort_cancel['Cancelled_Order'] > 0
)

cohort_customer_summary = (
    customer_cohort_cancel
    .groupby('Cohort_Month')
    .agg(
        Customers=('Customer ID', 'count'),
        Customers_With_Cancellation=('Cancelled_At_Least_Once', 'sum')
    )
    .reset_index()
)

cohort_customer_summary['Customer_Cancellation_Rate (%)'] = (
    cohort_customer_summary['Customers_With_Cancellation']
    / cohort_customer_summary['Customers']
    * 100
).round(2)

cohort_customer_summary

In [ ]:
customer_cohort_cancel = (
    df_30.groupby(['Cohort_Month', 'Customer ID'])
    .agg(
        Cancelled_Order=('Customer Cancelled',
                         lambda x: (x == 'Yes').sum())
    )
    .reset_index()
)

customer_cohort_cancel['Cancelled_At_Least_Once'] = (
    customer_cohort_cancel['Cancelled_Order'] > 0
)

cohort_customer_summary = (
    customer_cohort_cancel
    .groupby('Cohort_Month')
    .agg(
        Customers=('Customer ID', 'count'),
        Customers_With_Cancellation=('Cancelled_At_Least_Once', 'sum')
    )
    .reset_index()
)

cohort_customer_summary['Customer_Cancellation_Rate (%)'] = (
    cohort_customer_summary['Customers_With_Cancellation']
    / cohort_customer_summary['Customers']
    * 100
).round(2)

cohort_customer_summary

In [ ]:
import pandas as pd

cohort_cancel_table = pd.crosstab(
    customer_cohort_cancel['Cohort_Month'],
    customer_cohort_cancel['Cancelled_At_Least_Once']
)

# Make sure columns are False, True
cohort_cancel_table = cohort_cancel_table.reindex(
    columns=[False, True],
    fill_value=0
)

cohort_cancel_table

In [ ]:
from scipy.stats import fisher_exact

table = cohort_cancel_table.to_numpy()

result = fisher_exact(table)

print(result)

In [ ]:
import numpy as np

chi2, _, _, _ = chi2_contingency(
    table,
    correction=False
)

n = table.sum()
r, k = table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

print("Cramer's V:", round(cramers_v, 4))

## 6. Restaurant Operational Driver Analysis

In [ ]:
restaurant_ready_count = pd.crosstab(
    df['Restaurant name'],
    df['Order Ready Marked']
)

restaurant_ready_count

In [ ]:
restaurant_ready_pct = pd.crosstab(
    df['Restaurant name'],
    df['Order Ready Marked'],
    normalize='index'
) * 100

restaurant_ready_pct = restaurant_ready_pct.round(2)

restaurant_ready_pct

In [ ]:
restaurant_ready_pct['Incorrect + Missed (%)'] = (
    restaurant_ready_pct.get('Incorrectly', 0)
    + restaurant_ready_pct.get('Missed', 0)
).round(2)

restaurant_ready_pct = restaurant_ready_pct.sort_values(
    'Incorrect + Missed (%)',
    ascending=False
)

restaurant_ready_pct

In [ ]:
restaurant_volume = (
    df.groupby('Restaurant name')
      .size()
      .rename('Total Orders')
)

restaurant_ready_summary = (
    restaurant_ready_pct
    .join(restaurant_volume)
    .sort_values('Incorrect + Missed (%)', ascending=False)
)

restaurant_ready_summary

In [ ]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(
    df['Restaurant name'],
    df['Order Ready Marked']
)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square Statistic:", round(chi2, 4))
print("Degrees of Freedom:", dof)
print("P-value:", round(p_value, 4))
print("Degrees of Freedom:", dof)
print("Minimum Expected Frequency:", expected.min())
print("Expected Frequencies < 5:", (expected < 5).sum())

In [ ]:
from scipy.stats import fisher_exact, MonteCarloMethod

contingency_table = pd.crosstab(
    df['Restaurant name'],
    df['Order Ready Marked']
)

table = contingency_table.to_numpy()

result = fisher_exact(
    table,
    method=MonteCarloMethod(n_resamples=100000, rng=0)
)

print("Exact Monte Carlo p-value:", result.pvalue)

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

chi2, _, _, _ = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
r, k = contingency_table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

print("Cramer's V:", round(cramers_v, 4))

## 7. Cancellation & Revenue Impact Analysis

In [ ]:
import numpy as np
import pandas as pd

df_cancel = df[
    (df['Zomato_Cancelled'] == 'Yes') |
    (df['Customer Cancelled'] == 'Yes')
].copy()

In [ ]:
pd.crosstab(
    df_cancel['Zomato_Cancelled'],
    df_cancel['Customer Cancelled']
)

In [ ]:
df_cancel['Restaurant compensation (Cancellation)'].describe()

In [ ]:
df_cancel.groupby(
    ['Zomato_Cancelled', 'Customer Cancelled']
)['Restaurant compensation (Cancellation)'].agg(
    ['count', 'sum', 'mean', 'median', 'min', 'max']
)

In [ ]:
df_cancel.groupby(
    ['Zomato_Cancelled', 'Customer Cancelled']
)['Total'].agg(
    ['count', 'sum', 'mean', 'median', 'min', 'max']
)

In [ ]:
total_platform_order_value = df['Total'].sum()

print("Total platform order value:", total_platform_order_value)

print(
    "Cancelled order value share (%):",
    df_cancel['Total'].sum() /
    total_platform_order_value * 100
)

In [ ]:
import scikit_posthocs as sp

posthoc = sp.posthoc_dunn(
    df,
    val_col='KPT duration (minutes)',
    group_col='Time_of_Day',
    p_adjust='holm'
)

posthoc

## 8. Rider Wait Time, KPT & Customer Value Analysis

Does Rider wait time is associated with time of the day

In [ ]:
rider_wait_time_df = df[['Time_of_Day','Rider wait time (minutes)']].dropna()

In [ ]:
rider_wait_time_summary = (
    rider_wait_time_df
    .groupby('Time_of_Day')['Rider wait time (minutes)']
    .agg(
        Orders='count',
        Mean='mean',
        Median='median',
        Std='std'
    )
    .round(2)
)

rider_wait_time_summary

In [ ]:
from scipy.stats import kruskal

groups = [
    group['Rider wait time (minutes)'].dropna()
    for _, group in rider_wait_time_df.groupby('Time_of_Day')
]

h_stat, p_value = kruskal(*groups)

print("H-statistic:", h_stat)
print("p-value:", p_value)

In [ ]:
n = len(rider_wait_time_df)
k = rider_wait_time_df['Time_of_Day'].nunique()

epsilon_squared = (h_stat - k + 1) / (n - k)

print("Epsilon squared:", epsilon_squared)

In [ ]:
import scikit_posthocs as sp

posthoc_rider_wait = sp.posthoc_dunn(
    rider_wait_time_df,
    val_col='Rider wait time (minutes)',
    group_col='Time_of_Day',
    p_adjust='holm'
)

posthoc_rider_wait.round(6)

In [ ]:
kpt_time_df = df[['Time_of_Day', 'KPT duration (minutes)']].dropna()

In [ ]:
import scikit_posthocs as sp

posthoc_kpt = sp.posthoc_dunn(
    kpt_time_df,
    val_col='KPT duration (minutes)',
    group_col='Time_of_Day',
    p_adjust='holm'
)

posthoc_kpt.round(6)

In [ ]:
import pandas as pd

day_counts = df['day_of_week'].value_counts().sort_index()

expected = len(df) / day_counts.nunique()

contribution = ((day_counts - expected) ** 2) / expected

result = pd.DataFrame({
    'Observed Orders': day_counts,
    'Expected Orders': expected,
    'Chi-square Contribution': contribution
}).sort_values('Chi-square Contribution', ascending=False)

result

In [ ]:
month_counts = df['month_name'].value_counts().sort_index()

expected = len(df) / month_counts.nunique()

month_contribution = ((month_counts - expected) ** 2) / expected

month_result = pd.DataFrame({
    'Observed Orders': month_counts,
    'Expected Orders': expected,
    'Chi-square Contribution': month_contribution
}).sort_values('Chi-square Contribution', ascending=False)

month_result

In [ ]:
df = df.sort_values(['Customer ID', 'Order Date'])

df['Customer Order Number'] = (
    df.groupby('Customer ID').cumcount() + 1
)

In [ ]:
df[['Customer ID', 'Order Date', 'Customer Order Number']].head(20)

In [ ]:
order_number_counts = (
    df[df['Customer Order Number']>=6].groupby('Customer Order Number')['Customer ID']
      .nunique()
      .reset_index(name='Customers')
)

order_number_counts

In [ ]:
order_distribution = (
    df.groupby('Customer Order Number')['Customer ID']
      .nunique()
      .reset_index(name='Customers')
)

order_distribution.head(20)

In [ ]:
order_distribution['Cumulative Customers'] = (
    order_distribution['Customers'].cumsum()
)

order_distribution['Cumulative %'] = (
    order_distribution['Customers'].cumsum()
    / order_distribution['Customers'].sum() * 100
)

order_distribution.head(20)












order_distribution['Cumulative Customers'] = (
    order_distribution['Customers'].cumsum()
)

order_distribution['Cumulative %'] = (
    order_distribution['Customers'].cumsum()
    / order_distribution['Customers'].sum() * 100
)

order_distribution.head(20)

In [ ]:
order_distribution['% of Original Customers'] = (
    order_distribution['Customers'] / 11607 * 100
)

order_distribution.head(20)

In [ ]:
def ordinal(n):
    if n == 1:
        return '1st order'
    elif n == 2:
        return '2nd order'
    elif n == 3:
        return '3rd order'
    else:
        return f'{n}th order'

def order_group(x):
    if x <= 5:
        return ordinal(x)
    elif x <= 10:
        return '6th–10th order'
    else:
        return '11th+ order'

df['Order Group'] = df['Customer Order Number'].apply(order_group)

In [ ]:
df['Order Group'].value_counts()

In [ ]:
# checking kpt
kpt_df = df.dropna(subset=['KPT duration (minutes)']).copy()

kpt_by_order = (
    kpt_df.groupby('Order Group')['KPT duration (minutes)']
    .agg(
        Orders='count',
        Mean='mean',
        Median='median',
        Std='std'
    )
)

order = [
    '1st order',
    '2nd order',
    '3rd order',
    '4th order',
    '5th order',
    '6th–10th order',
    '11th+ order'
]

kpt_by_order = kpt_by_order.reindex(order)

kpt_by_order

In [ ]:
order = [
    '1st order',
    '2nd order',
    '3rd order',
    '4th order',
    '5th order',
    '6th–10th order',
    '11th+ order'
]

groups = [
    kpt_df.loc[kpt_df['Order Group'] == g, 'KPT duration (minutes)'].values
    for g in order
]

h_stat, p_value = kruskal(*groups)

print("H-statistic:", h_stat)
print("p-value:", p_value)

In [ ]:
n = len(kpt_df)
k = len(order)

epsilon_squared = (h_stat - k + 1) / (n - k)

print("Epsilon squared:", epsilon_squared)

In [ ]:
import scikit_posthocs as sp

dunn_result = sp.posthoc_dunn(
    kpt_df,
    val_col='KPT duration (minutes)',
    group_col='Order Group',
    p_adjust='bonferroni'
)

dunn_result

In [ ]:
# checking for rider wait time
# Remove rows where Rider Wait Time is missing
wait_df = df.dropna(subset=['Rider wait time (minutes)']).copy()

order = [
    '1st order',
    '2nd order',
    '3rd order',
    '4th order',
    '5th order',
    '6th–10th order',
    '11th+ order'
]

# Descriptive statistics
wait_by_order = (
    wait_df.groupby('Order Group')['Rider wait time (minutes)']
    .agg(
        Orders='count',
        Mean='mean',
        Median='median',
        Std='std'
    )
    .reindex(order)
)

wait_by_order

In [ ]:
from scipy.stats import kruskal

order = [
    '1st order',
    '2nd order',
    '3rd order',
    '4th order',
    '5th order',
    '6th–10th order',
    '11th+ order'
]

groups = [
    wait_df.loc[wait_df['Order Group'] == g, 'Rider wait time (minutes)'].values
    for g in order
]

h_stat, p_value = kruskal(*groups)

print("H-statistic:", h_stat)
print("p-value:", p_value)

In [ ]:
n = len(wait_df)
k = len(order)

epsilon_squared = (h_stat - k + 1) / (n - k)

print("Epsilon squared:", epsilon_squared)

In [ ]:
# checking total discount
discount_df = df.dropna(subset=['Total Discount']).copy()

discount_by_order = (
    discount_df.groupby('Order Group')['Total Discount']
    .agg(
        Orders='count',
        Mean='mean',
        Median='median',
        Std='std'
    )
    .reindex(order)
)

discount_by_order

In [ ]:
from scipy.stats import kruskal

order = [
    '1st order',
    '2nd order',
    '3rd order',
    '4th order',
    '5th order',
    '6th–10th order',
    '11th+ order'
]

groups = [
    discount_df.loc[
        discount_df['Order Group'] == g,
        'Total Discount'
    ].values
    for g in order
]

h_stat, p_value = kruskal(*groups)

print("H-statistic:", h_stat)
print("p-value:", p_value)

In [ ]:
n = len(discount_df)
k = len(order)

epsilon_squared = (h_stat - k + 1) / (n - k)

print("Epsilon squared:", epsilon_squared)

In [ ]:
import scikit_posthocs as sp

dunn_result = sp.posthoc_dunn(
    discount_df,
    val_col='Total Discount',
    group_col='Order Group',
    p_adjust='bonferroni'
)

dunn_result

## 9. Four-Week Retention Analysis & First-Order Drivers

In [ ]:
# different question
df['Order Date'] = pd.to_datetime(df['Order Date'])

first_order = (
    df.groupby('Customer ID')['Order Date']
      .min()
      .reset_index(name='First Order Date')
)

first_order.head()

In [ ]:
study_end = df['Order Date'].max()

eligible_customers = first_order[
    first_order['First Order Date'] <= study_end - pd.Timedelta(weeks=4)
].copy()

print("Total customers:", len(first_order))
print("Eligible customers:", len(eligible_customers))

In [ ]:
df_retention = df.merge(
    eligible_customers,
    on='Customer ID',
    how='inner'
)

In [ ]:
df_retention['Days Since First Order'] = (
    df_retention['Order Date'] - df_retention['First Order Date']
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
df_followup = df_retention[
    (df_retention['Days Since First Order'] > 0) &
    (df_retention['Days Since First Order'] <= 28)
].copy()

In [ ]:
returned_customers = (
    df_followup.groupby('Customer ID')
    .size()
    .reset_index(name='Repeat Orders')
)

returned_customers['Returned Within 4 Weeks'] = True

In [ ]:
customer_retention = eligible_customers.merge(
    returned_customers[['Customer ID', 'Returned Within 4 Weeks']],
    on='Customer ID',
    how='left'
)

customer_retention['Returned Within 4 Weeks'] = (
    customer_retention['Returned Within 4 Weeks']
    .fillna(False)
)

In [ ]:
customer_retention['Returned Within 4 Weeks'].value_counts()

In [ ]:
delivered_df = df[df['Order Status'] == 'Delivered'].copy()

delivered_df['Order Date'] = pd.to_datetime(delivered_df['Order Date'])


first_delivered = (
    delivered_df.groupby('Customer ID')['Order Date']
    .min()
    .reset_index(name='First Delivered Order Date')
)


study_end = delivered_df['Order Date'].max()

eligible_customers = first_delivered[
    first_delivered['First Delivered Order Date']
    <= study_end - pd.Timedelta(weeks=4)
].copy()




delivered_retention = delivered_df.merge(
    eligible_customers,
    on='Customer ID',
    how='inner'
)



delivered_retention['Days Since First Order'] = (
    delivered_retention['Order Date']
    - delivered_retention['First Delivered Order Date']
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
followup_delivered = delivered_retention[
    (delivered_retention['Days Since First Order'] > 0) &
    (delivered_retention['Days Since First Order'] <= 28)
].copy()

In [ ]:
repeat_customers = (
    followup_delivered.groupby('Customer ID')
    .size()
    .reset_index(name='Additional Delivered Orders')
)

repeat_customers['Returned Successfully Within 4 Weeks'] = True

In [ ]:
customer_retention = eligible_customers.merge(
    repeat_customers[
        ['Customer ID', 'Returned Successfully Within 4 Weeks']
    ],
    on='Customer ID',
    how='left'
)

customer_retention['Returned Successfully Within 4 Weeks'] = (
    customer_retention['Returned Successfully Within 4 Weeks']
    .fillna(False)
)

In [ ]:
customer_retention[
    'Returned Successfully Within 4 Weeks'
].value_counts()

In [ ]:
first_orders = (
    delivered_df.sort_values(['Customer ID', 'Order Date'])
    .groupby('Customer ID')
    .first()
    .reset_index()
)

first_orders = first_orders.merge(
    customer_retention[
        ['Customer ID', 'Returned Successfully Within 4 Weeks']
    ],
    on='Customer ID',
    how='left'
)

In [ ]:
first_orders[
    'Returned Successfully Within 4 Weeks'
].value_counts()

In [ ]:
kpt_first = first_orders.dropna(subset=['KPT duration (minutes)']).copy()

kpt_first.groupby(
    'Returned Successfully Within 4 Weeks'
)['KPT duration (minutes)'].agg(
    Orders='count',
    Mean='mean',
    Median='median',
    Std='std'
)

In [ ]:
from scipy.stats import mannwhitneyu

group_false = kpt_first.loc[
    kpt_first['Returned Successfully Within 4 Weeks'] == False,
    'KPT duration (minutes)'
]

group_true = kpt_first.loc[
    kpt_first['Returned Successfully Within 4 Weeks'] == True,
    'KPT duration (minutes)'
]

u_stat, p_value = mannwhitneyu(
    group_false,
    group_true,
    alternative='two-sided'
)

print("U-statistic:", u_stat)
print("p-value:", p_value)

In [ ]:
# checking rider wait time
wait_first = first_orders.dropna(subset=['Rider wait time (minutes)']).copy()

wait_first.groupby(
    'Returned Successfully Within 4 Weeks'
)['Rider wait time (minutes)'].agg(
    Orders='count',
    Mean='mean',
    Median='median',
    Std='std'
)

In [ ]:
from scipy.stats import mannwhitneyu

group_false = wait_first.loc[
    wait_first['Returned Successfully Within 4 Weeks'] == False,
    'Rider wait time (minutes)'
]

group_true = wait_first.loc[
    wait_first['Returned Successfully Within 4 Weeks'] == True,
    'Rider wait time (minutes)'
]

u_stat, p_value = mannwhitneyu(
    group_false,
    group_true,
    alternative='two-sided'
)

print("U-statistic:", u_stat)
print("p-value:", p_value)

In [ ]:
discount_first = first_orders.dropna(
    subset=['Total Discount']
).copy()

discount_first.groupby(
    'Returned Successfully Within 4 Weeks'
)['Total Discount'].agg(
    Orders='count',
    Mean='mean',
    Median='median',
    Std='std'
)

In [ ]:
from scipy.stats import mannwhitneyu

group_false = discount_first.loc[
    discount_first['Returned Successfully Within 4 Weeks'] == False,
    'Total Discount'
]

group_true = discount_first.loc[
    discount_first['Returned Successfully Within 4 Weeks'] == True,
    'Total Discount'
]

u_stat, p_value = mannwhitneyu(
    group_false,
    group_true,
    alternative='two-sided'
)

print("U-statistic:", u_stat)
print("p-value:", p_value)

In [ ]:
order_value_first = first_orders.dropna(
    subset=['Total']
).copy()

order_value_first.groupby(
    'Returned Successfully Within 4 Weeks'
)['Total'].agg(
    Orders='count',
    Mean='mean',
    Median='median',
    Std='std'
)

In [ ]:
from scipy.stats import mannwhitneyu

group_false = order_value_first.loc[
    order_value_first['Returned Successfully Within 4 Weeks'] == False,
    'Total'
]

group_true = order_value_first.loc[
    order_value_first['Returned Successfully Within 4 Weeks'] == True,
    'Total'
]

u_stat, p_value = mannwhitneyu(
    group_false,
    group_true,
    alternative='two-sided'
)

print("U-statistic:", u_stat)
print("p-value:", p_value)

In [ ]:
n1 = len(group_false)
n2 = len(group_true)

rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print("Rank-biserial correlation:", rank_biserial)

## 10. Customer Cohort & High-Value Customer Analysis

In [ ]:
# Add cohort month based on the first delivered order
eligible_customers['Cohort'] = (
    eligible_customers['First Delivered Order Date']
    .dt.to_period('M')
    .astype(str)
)

# Customers who successfully returned within 4 weeks
repeat_customer_ids = set(
    repeat_customers['Customer ID']
)

# Mark whether each eligible customer returned successfully
eligible_customers['Returned Within 4 Weeks'] = (
    eligible_customers['Customer ID']
    .isin(repeat_customer_ids)
)

# Cohort-level retention
cohort_retention = (
    eligible_customers
    .groupby('Cohort')
    .agg(
        Customers=('Customer ID', 'count'),
        Returned_Customers=('Returned Within 4 Weeks', 'sum')
    )
)

cohort_retention['4_Week_Return_Rate (%)'] = (
    cohort_retention['Returned_Customers']
    / cohort_retention['Customers']
    * 100
)

cohort_retention

In [ ]:
# checking conditions

first_orders['Cohort'] = (
    first_orders['Order Date']
    .dt.to_period('M')
    .astype(str)
)


cohort_conditions = (
    first_orders
    .groupby('Cohort')
    .agg(
        Customers=('Customer ID', 'count'),
        Median_KPT=('KPT duration (minutes)', 'median'),
        Median_Rider_Wait=('Rider wait time (minutes)', 'median'),
        Median_Discount=('Total Discount', 'median'),
        Median_Order_Value=('Total', 'median')
    )
)

first_orders_cohort = first_orders[
    first_orders['Customer ID'].isin(
        eligible_customers['Customer ID']
    )
].copy()



cohort_conditions = (
    first_orders_cohort
    .groupby('Cohort')
    .agg(
        Customers=('Customer ID', 'count'),
        Median_KPT=('KPT duration (minutes)', 'median'),
        Median_Rider_Wait=('Rider wait time (minutes)', 'median'),
        Median_Discount=('Total Discount', 'median'),
        Median_Order_Value=('Total', 'median')
    )
)

cohort_conditions

In [ ]:
time_of_day_cohort = pd.crosstab(
    first_orders_cohort['Cohort'],
    first_orders_cohort['Time_of_Day'],
    normalize='index'
) * 100

time_of_day_cohort.round(2)

In [ ]:
day_of_week_cohort = pd.crosstab(
    first_orders_cohort['Cohort'],
    first_orders_cohort['day_of_week'],
    normalize='index'
) * 100

day_of_week_cohort.round(2)

In [ ]:
# Keep successfully delivered orders
delivered = df[df['Order Status'] == 'Delivered'].copy()

# Total spending per customer
customer_spend = (
    delivered.groupby('Customer ID')['Total']
    .sum()
    .reset_index(name='Total Spend')
)

# Top 20% spending customers
cutoff = customer_spend['Total Spend'].quantile(0.80)

customer_spend['Customer Segment'] = np.where(
    customer_spend['Total Spend'] >= cutoff,
    'Top 20%',
    'Remaining 80%'
)

customer_spend['Customer Segment'].value_counts()

In [ ]:
# Merge customer segment back to delivered orders
delivered = delivered.merge(
    customer_spend[['Customer ID', 'Customer Segment']],
    on='Customer ID',
    how='left'
)

# Sort orders chronologically
delivered = delivered.sort_values(
    ['Customer ID', 'Order Date']
)

# Previous order date for each customer
delivered['Previous Order Date'] = (
    delivered.groupby('Customer ID')['Order Date']
    .shift(1)
)

# Reorder interval in days
delivered['Reorder Interval'] = (
    delivered['Order Date'] - delivered['Previous Order Date']
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
customer_reorder = (
    delivered.dropna(subset=['Reorder Interval'])
    .groupby(['Customer ID', 'Customer Segment'])
    .agg(
        Reorder_Interval=('Reorder Interval', 'mean')
    )
    .reset_index()
)

In [ ]:
customer_reorder.groupby('Customer Segment')['Reorder_Interval'].agg(
    Customers='count',
    Mean='mean',
    Median='median',
    Std='std'
)

In [ ]:
from scipy.stats import mannwhitneyu

top20 = customer_reorder.loc[
    customer_reorder['Customer Segment'] == 'Top 20%',
    'Reorder_Interval'
]

remaining80 = customer_reorder.loc[
    customer_reorder['Customer Segment'] == 'Remaining 80%',
    'Reorder_Interval'
]

u_stat, p_value = mannwhitneyu(
    top20,
    remaining80,
    alternative='two-sided'
)

print("U-statistic:", u_stat)
print("p-value:", p_value)

In [ ]:
n1 = len(top20)
n2 = len(remaining80)

rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print("Rank-biserial correlation:", rank_biserial)

In [ ]:
# Successfully delivered orders only
delivered = df[df['Order Status'] == 'Delivered'].copy()

# Customer-level metrics
customer_metrics = (
    delivered.groupby('Customer ID')
    .agg(
        Total_Spend=('Total', 'sum'),
        Order_Count=('Order ID', 'nunique'),
        Average_Order_Value=('Total', 'mean')
    )
    .reset_index()
)

# Define Top 20% based on total spending
cutoff = customer_metrics['Total_Spend'].quantile(0.80)

customer_metrics['Customer Segment'] = np.where(
    customer_metrics['Total_Spend'] >= cutoff,
    'Top 20%',
    'Remaining 80%'
)

customer_metrics.head()

In [ ]:
customer_metrics.groupby('Customer Segment').agg(
    Customers=('Customer ID', 'count'),
    Median_Orders=('Order_Count', 'median'),
    Median_AOV=('Average_Order_Value', 'median')
)

In [ ]:
from scipy.stats import mannwhitneyu

top20_orders = customer_metrics.loc[
    customer_metrics['Customer Segment'] == 'Top 20%',
    'Order_Count'
]

remaining80_orders = customer_metrics.loc[
    customer_metrics['Customer Segment'] == 'Remaining 80%',
    'Order_Count'
]

u_orders, p_orders = mannwhitneyu(
    top20_orders,
    remaining80_orders,
    alternative='two-sided'
)

print("U-statistic:", u_orders)
print("p-value:", p_orders)

In [ ]:
n1 = len(top20_orders)
n2 = len(remaining80_orders)

effect_orders = 1 - (2 * u_orders) / (n1 * n2)

print("Rank-biserial correlation:", effect_orders)

In [ ]:
top20_aov = customer_metrics.loc[
    customer_metrics['Customer Segment'] == 'Top 20%',
    'Average_Order_Value'
]

remaining80_aov = customer_metrics.loc[
    customer_metrics['Customer Segment'] == 'Remaining 80%',
    'Average_Order_Value'
]

u_aov, p_aov = mannwhitneyu(
    top20_aov,
    remaining80_aov,
    alternative='two-sided'
)

print("U-statistic:", u_aov)
print("p-value:", p_aov)

In [ ]:
n1 = len(top20_aov)
n2 = len(remaining80_aov)

effect_aov = 1 - (2 * u_aov) / (n1 * n2)

print("Rank-biserial correlation:", effect_aov)

## 11. Final Analytical Outputs

The tables and statistical results generated above are used as supporting analysis for the final business case study and Power BI dashboard.

The dashboard itself was built separately in Power BI. This notebook is intentionally focused on the underlying Python analysis rather than reproducing the dashboard visuals.
